# 07 - Exploration des Données Images

## Section : Gestion des Images

### Classification de Produits E-commerce

## Objectif
Explorer le jeu d'images produits pour :
- La **structure** du dataset (chemins, formats, convention de nommage)
- La **couverture** (produits avec/sans image)
- La **distribution** des classes par image
- Les **caractéristiques** visuelles (tailles, formats, qualité)
- La **détection** d'images potentiellement corrompues

## Plan
1. Chargement et inventaire des images
2. Analyse de la couverture produit / image
3. Distribution des classes
4. Caractéristiques des images (tailles, formats)
5. Échantillons visuels par classe
6. Détection d'anomalies (fichiers corrompus)
7. Synthèse et recommandations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append(str(Path('../').resolve()))

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

ROOT = Path('../').resolve()
DATA_BRUT = ROOT / 'data brut'
IMAGE_DIR = ROOT / 'data' / 'processed' / 'image_train'

print("✅ Bibliothèques importées")
print(f"📁 Images : {IMAGE_DIR}")

## 1. Inventaire des Images

In [ ]:
image_files = list(IMAGE_DIR.glob('*.jpg'))
print(f"📷 Nombre total de fichiers .jpg : {len(image_files):,}")
print(f"   Convention : image_{{imageid}}_product_{{productid}}.jpg")

image_data = []
for f in image_files:
    try:
        parts = f.stem.split('_')
        if len(parts) >= 4 and parts[0] == 'image' and parts[2] == 'product':
            image_data.append({'imageid': int(parts[1]), 'productid': int(parts[3]), 'path': str(f)})
    except (ValueError, IndexError):
        pass

image_df = pd.DataFrame(image_data)
print(f"\n✅ Images parsées : {len(image_df):,}")
print(f"   Produits uniques : {image_df['productid'].nunique():,}")
image_df.head(10)

## 2. Couverture Produit / Image

In [ ]:
X_train = pd.read_csv(DATA_BRUT / 'X_train_update.csv', index_col=0)
y_train = pd.read_csv(DATA_BRUT / 'Y_train.csv', index_col=0)

merged_cov = X_train.merge(image_df[['productid', 'imageid']], on=['productid', 'imageid'], how='inner')
covered = len(merged_cov)
total = len(X_train)

print(f"📊 Couverture :")
print(f"   Produits totaux (train) : {total:,}")
print(f"   Produits avec image : {covered:,} ({100*covered/total:.1f}%)")
print(f"   Produits sans image : {total - covered:,}")

## 3. Distribution des Classes

In [ ]:
merged = X_train.merge(image_df[['productid', 'imageid']], on=['productid', 'imageid'], how='inner')
merged = merged.merge(y_train, left_index=True, right_index=True)

class_dist = merged['prdtypecode'].value_counts().sort_index()
print(f"📊 Distribution par classe (27 classes) :")
print(class_dist.head(10))

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(class_dist)), class_dist.values)
ax.set_xlabel('Classe (prdtypecode)')
ax.set_ylabel('Nombre d\'images')
ax.set_title('Distribution des images par classe')
plt.xticks(range(len(class_dist)), class_dist.index, rotation=45)
plt.tight_layout()
plt.show()

## 4. Caractéristiques des Images (tailles, formats)

In [ ]:
from PIL import Image

sample_size = min(500, len(image_df))
sample = image_df.sample(n=sample_size, random_state=42)

sizes = []
for _, row in sample.iterrows():
    try:
        with Image.open(row['path']) as img:
            sizes.append({'w': img.width, 'h': img.height})
    except Exception:
        sizes.append({'w': None, 'h': None})

sizes_df = pd.DataFrame(sizes)
valid = sizes_df.dropna()
print(f"📐 Échantillon de {len(valid)} images :")
print(f"   Largeur : min={valid['w'].min()}, max={valid['w'].max()}, médiane={valid['w'].median():.0f}")
print(f"   Hauteur : min={valid['h'].min()}, max={valid['h'].max()}, médiane={valid['h'].median():.0f}")
print(f"   Corrompues / illisibles : {len(sizes_df) - len(valid)}")

## 5. Échantillons Visuels par Classe

In [ ]:
df_with_class = merged.merge(image_df[['productid', 'imageid', 'path']], on=['productid', 'imageid'])

n_classes_show = 6
samples_per_class = 4
classes = df_with_class['prdtypecode'].value_counts().head(n_classes_show).index.tolist()

fig, axes = plt.subplots(n_classes_show, samples_per_class, figsize=(12, 14))
for i, cls in enumerate(classes):
    sub = df_with_class[df_with_class['prdtypecode'] == cls].sample(n=samples_per_class, random_state=42)
    for j, (_, row) in enumerate(sub.iterrows()):
        try:
            img = Image.open(row['path']).convert('RGB')
            axes[i, j].imshow(img)
        except Exception:
            axes[i, j].text(0.5, 0.5, 'Erreur', ha='center', va='center')
        axes[i, j].set_title(f"Classe {cls}", fontsize=9)
        axes[i, j].axis('off')
plt.suptitle('Échantillons d\'images par classe', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Synthèse et Recommandations

In [ ]:
print("📋 Synthèse de l'exploration :")
print(f"   • Images disponibles : {len(image_df):,}")
print(f"   • Couverture produits : {100*covered/total:.1f}%")
print(f"   • 27 classes représentées")
print(f"   • Format attendu : redimensionnement 224×224 pour ResNet")
print("\n✅ Prochaine étape : Notebook 08 - Traitement des images")